# Explore multi-tissue Xenium outputs

The 8 Xenium outputs whose `main`-tab rows share one `file name` each bundle **2 FH subjects** in a single scanned region (TMA cores). Before building any split pipeline, this notebook lets us **look at each one**: plot cell centroids and try a **DBSCAN** spatial clustering to see whether the two cores separate cleanly.

**Read-only** — it does not modify or write any output bundle. Only `cells.parquet` (~1.3 MB each) is downloaded, not the multi-GB bundles.

## Config & inventory (driven by the tracking sheet)

In [ ]:
import pandas as pd
from pathlib import Path

REPO = Path("/home/workspace/PCD_data/xenium")
XLSX = REPO / "data" / "xenium data tracking.xlsx"
CELLS_DIR = REPO / "data" / "multitissue_cells"   # local, git-ignored
BUCKET = "gs://temp-xenium-hise-transfer"

main = pd.read_excel(XLSX, sheet_name="main")
main.columns = [str(c).strip() for c in main.columns]

# rows flagged for separation (the 'Y' we wrote into the sheet)
sep = main[main["multiple samples"].astype(str).str.upper().str.startswith("Y")].copy()

# one output per unique file name; the 2 rows per output are the 2 subjects
groups = (sep.groupby("file name")
             .agg(slide=("slide id", "first"),
                  exp=("experiment id", "first"),
                  section=("section id", "first"),
                  block=("block id", "first"),
                  samples=("potential sample ids (left, right)", "first"),
                  subjects=("cohort", lambda s: sorted(s.dropna().astype(str).unique())))
             .reset_index())

# temp-bucket folder = basename of the file name, minus .zip
groups["folder"] = groups["file name"].str.split("/").str[-1].str.replace(".zip", "", regex=False)
groups["label"] = groups["section"].astype(str)

print(f"{len(groups)} multi-tissue outputs\n")
for _, r in groups.iterrows():
    print(f"  {r.label:10s} block {r.block}  slide {r.slide}  {r.exp}  ->  {' + '.join(r.subjects)}")

## Download `cells.parquet` for each output

For each output: try the temp GCS bucket first (`gcloud storage cp` — cheap, grabs only `cells.parquet`). **`TMA00142`** (slide 0031088) isn't in that bucket — it lives in HISE only, stored as one zip in the `PCD_Spatial` project store — so it falls back to a **HISE fetch**: download the bundle into the read-only `~/input/` cache and extract just `cells.parquet` (nothing large hits writable disk).

The HISE step shells out to the **`minimal`** conda env's Python (which has `hisepy`) so this analysis env stays minimal. HISE auth comes from the instance identity, so the env doesn't matter — but your session must have HISE access.

In [ ]:
import subprocess

CELLS_DIR.mkdir(parents=True, exist_ok=True)
MINIMAL_PY = "/home/workspace/environment/minimal/bin/python"  # has hisepy; HISE auth is instance-based

# HISE fallback: the whole output is one zip in the PCD_Spatial store; download it into the
# read-only ~/input/ cache and extract just cells.parquet (nothing large hits writable disk).
HISE_FETCH = r'''
import glob, os, sys, zipfile
import hisepy as hp
store, file_name, out = sys.argv[1:4]
hp.download_from_project_store(store, file_name)
base = os.path.basename(file_name)
zp = max(glob.glob(os.path.expanduser("~/input/**/" + base), recursive=True), key=os.path.getmtime)
os.makedirs(os.path.dirname(out), exist_ok=True)
with zipfile.ZipFile(zp) as z:
    m = next(n for n in z.namelist() if os.path.basename(n) == "cells.parquet")
    with z.open(m) as s, open(out, "wb") as d:
        d.write(s.read())
'''

for _, r in groups.iterrows():
    dst = CELLS_DIR / f"{r.label}.parquet"
    if dst.exists():
        print(f"have   {r.label}")
        continue
    # 1) temp GCS bucket (cheap — only cells.parquet)
    res = subprocess.run(["gcloud", "storage", "cp",
                          f"{BUCKET}/{r.folder}/cells.parquet", str(dst)],
                         capture_output=True, text=True)
    if res.returncode == 0:
        print(f"pulled {r.label} (bucket)")
        continue
    # 2) HISE fallback (e.g. TMA00142 — not in the bucket)
    print(f"{r.label} not in bucket -> HISE ...")
    res = subprocess.run([MINIMAL_PY, "-c", HISE_FETCH, "PCD_Spatial", r["file name"], str(dst)],
                         capture_output=True, text=True)
    if res.returncode == 0 and dst.exists():
        print(f"pulled {r.label} (HISE)")
    else:
        tail = (res.stderr.strip().splitlines() or [""])[-1]
        print(f"FAILED {r.label}: {tail}")

## Install scikit-learn (for DBSCAN) if needed

In [ ]:
try:
    import sklearn
    print("scikit-learn present:", sklearn.__version__)
except ImportError:
    # installs into the active kernel; `conda install -c conda-forge scikit-learn` also works
    %pip install -q scikit-learn

## Raw spatial layout — FFPE block photo vs. cell centroids

One row per output: **left** = the FFPE block photo (from `Fred Hutch FFPE blocks.zip`, matched by block id, with the `L`/`R` sample ids marked per the block's left→right orientation), **right** = the Xenium cell centroids. Compare the orientation of the physical block against the centroid layout before deciding how the two cores should split.

The two sample ids are also drawn on each centroid panel so you can label which core is which. **Determine that manually, then nudge the positions in `SAMPLE_LABEL_POS`** — coordinates are axes fraction (`(0,0)` = bottom-left, `(1,1)` = top-right); each sample defaults to its left/right side.

In [ ]:
import io, zipfile
import matplotlib.pyplot as plt
from PIL import Image

BLOCKS_ZIP = REPO / "data" / "Fred Hutch FFPE blocks.zip"

# Sample-id labels drawn onto the centroid panels, for manual placement.
# Coords are AXES FRACTION: (0,0)=bottom-left, (1,1)=top-right of the panel.
# Nudge each sample id until it sits over its core. Left sample defaults left, right defaults right.
SAMPLE_LABEL_POS = {
    "TMA00238": {"TIS02600-001": (0.2, 0.25), "TIS02435-001": (0.8, 0.6)},
    "TMA00240": {"TIS06483-001": (0.25, 0.5), "TIS02619-001": (0.88, 0.85)},
    "TMA00244": {"TIS04965-001": (0.25, 0.25), "TIS02618-001": (0.75, 0.9)},
    "TMA00246": {"TIS02599-001": (0.18, 0.35), "TIS02429-001": (0.8, 0.525)},
    "TMA00142": {"TIS04397-001": (0.9, 0.35), "TIS04966-001": (0.2, 0.7)},
    "TMA00145": {"TIS02439-001": (0.25, 0.6), "TIS04436-001": (0.85, 0.5)},
    "TMA00161": {"TIS02608-001": (0.25, 0.3), "TIS02424-001": (0.75, 0.8)},
    "TMA00157": {"TIS04436-001": (0.25, 0.2), "TIS02439-001": (0.75, 0.7)},
    "TMA00159": {"TIS02609-001": (0.2, 0.15), "TIS02594-001": (0.75, 0.5)},
}

def load_cells(label):
    p = CELLS_DIR / f"{label}.parquet"
    if not p.exists():
        return None
    return pd.read_parquet(p, columns=["cell_id", "x_centroid", "y_centroid"])

def load_block(block_id):
    """FFPE block photo for a block id, read straight from the zip (matched by basename stem)."""
    if not BLOCKS_ZIP.exists():
        return None
    with zipfile.ZipFile(BLOCKS_ZIP) as z:
        name = next((n for n in z.namelist()
                     if Path(n).stem == block_id and n.lower().endswith(".jpg")), None)
        if name is None:
            return None
        with z.open(name) as f:
            return Image.open(io.BytesIO(f.read())).copy()

labels_list = groups["label"].tolist()
fig, axes = plt.subplots(len(labels_list), 2, figsize=(11, 4.5 * len(labels_list)))
for (ax_pts, ax_img), lab in zip(axes, labels_list):
    row = groups[groups.label == lab].iloc[0]
    subs = " + ".join(row.subjects)
    lr = [s.strip() for s in str(row.samples).split(",")] if pd.notna(row.samples) else []

    # left: FFPE block photo (external, keyed by block id)
    img = load_block(row.block)
    if img is None:
        ax_img.set_title(f"{row.block} — block photo MISSING")
    else:
        ax_img.imshow(img)
        ax_img.set_title(f"{row.block} — FFPE block")
        # label the left/right sample ids on the photo (each block is oriented left -> right)
        if len(lr) == 2:
            box = dict(boxstyle="round", fc="black", alpha=0.55, ec="none")
            ax_img.text(0.15, 0.75, f"L: {lr[0]}", transform=ax_img.transAxes,
                        ha="left", va="top", fontsize=8, color="white", bbox=box)
            ax_img.text(0.85, 0.75, f"R: {lr[1]}", transform=ax_img.transAxes,
                        ha="right", va="top", fontsize=8, color="white", bbox=box)
    ax_img.axis("off")

    # right: Xenium cell centroids
    df = load_cells(lab)
    if df is None:
        ax_pts.set_title(f"{lab} · {row.block} — MISSING (not downloaded)"); ax_pts.axis("off"); continue
    s = df.sample(60000, random_state=0) if len(df) > 60000 else df
    ax_pts.scatter(s.x_centroid, s.y_centroid, s=1, alpha=0.3, linewidths=0)
    ax_pts.set_aspect("equal"); ax_pts.invert_yaxis()
    ax_pts.set_title(f"{lab} · {row.block}  ({subs})  n={len(df):,}")
    ax_pts.set_xlabel("x (µm)"); ax_pts.set_ylabel("y (µm)")

    # sample-id labels for manual placement onto the two cores (positions from SAMPLE_LABEL_POS)
    default_xy = [(0.25, 0.5), (0.75, 0.5)]  # left sample, right sample
    lblbox = dict(boxstyle="round", fc="white", alpha=0.7, ec="0.4")
    for i, sid in enumerate(lr):
        x, y = SAMPLE_LABEL_POS.get(lab, {}).get(sid, default_xy[i] if i < 2 else (0.5, 0.5))
        ax_pts.text(x, y, sid, transform=ax_pts.transAxes, ha="center", va="center",
                    fontsize=8, color="black", bbox=lblbox)

plt.tight_layout()
plt.subplots_adjust(wspace=0.05)   # tighten the col1 <-> col2 gap
plt.show()

## DBSCAN separation — all 8 outputs

Tune `EPS_DEFAULT` / `MIN_SAMPLES` globally, or override a single output in `EPS_OVERRIDE`. `eps` is in **microns**. Noise points (label `-1`) are grey. Goal: **2 dense clusters ≈ the 2 subjects.** If a core is over-fragmented, raise `eps`; if the two cores merge into one cluster, lower `eps`.

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN

EPS_DEFAULT = 100      # microns
MIN_SAMPLES = 20
EPS_OVERRIDE = {}      # e.g. {"TMA00240": 150}

def run_dbscan(df, eps, min_samples):
    xy = df[["x_centroid", "y_centroid"]].to_numpy()
    return DBSCAN(eps=eps, min_samples=min_samples).fit_predict(xy)

# compute once per output
results = {}
for lab in labels_list:
    df = load_cells(lab)
    if df is None:
        results[lab] = None; continue
    eps = EPS_OVERRIDE.get(lab, EPS_DEFAULT)
    results[lab] = df.assign(cluster=run_dbscan(df, eps, MIN_SAMPLES))

# plot
n_rows = (len(labels_list) + 1) // 2
fig, axes = plt.subplots(n_rows, 2, figsize=(12, 5 * n_rows))
for ax, lab in zip(axes.ravel(), labels_list):
    row = groups[groups.label == lab].iloc[0]
    subs = " + ".join(row.subjects)
    df = results[lab]
    if df is None:
        ax.set_title(f"{lab} — MISSING"); ax.axis("off"); continue
    eps = EPS_OVERRIDE.get(lab, EPS_DEFAULT)
    n_real = (df.cluster.unique() >= 0).sum()
    for c, sub in df.groupby("cluster"):
        color = "0.8" if c == -1 else None
        s = sub.sample(40000, random_state=0) if len(sub) > 40000 else sub
        lbl = "noise" if c == -1 else f"c{c} ({len(sub):,})"
        ax.scatter(s.x_centroid, s.y_centroid, s=1, alpha=0.4, linewidths=0, color=color, label=lbl)
    ax.set_aspect("equal"); ax.invert_yaxis()
    ax.set_title(f"{lab}  ({subs})  eps={eps}  clusters={n_real}")
    ax.legend(markerscale=6, fontsize=7, loc="right")
# hide any unused axes (when len(labels_list) is odd)
for ax in axes.ravel()[len(labels_list):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

# cluster-size summary
print("cluster sizes (label: n_cells; -1 = noise)\n")
for lab in labels_list:
    df = results[lab]
    if df is None:
        print(f"{lab:10s} MISSING"); continue
    vc = df.cluster.value_counts().sort_index()
    print(f"{lab:10s} " + "  ".join(f"{int(k)}:{int(v):,}" for k, v in vc.items()))

## Next steps

Once we can see the two cores separate cleanly (2 dominant clusters per output), the plan is **DBSCAN → subset per-cell / per-transcript files** into per-tissue outputs ("option B"). This notebook is only for confirming the separation is sound before building that pipeline.